## MC2 Python Data Cleaning

### Read Data

In [32]:
# Core imports
import pandas as pd
import numpy as np
from datetime import datetime
import os

# Display all columns when debugging
pd.set_option('display.max_columns', None)

# Read data
cc_data_path = 'cc_data.csv'
loyalty_data_path = 'loyalty_data.csv'
gps_data_path = 'gps.csv'
car_assignments_path = 'car-assignments.csv'

# Read the CSVs
cc_data = pd.read_csv(cc_data_path, encoding='latin1')         # Using latin1 encoding to fix Unicode errors
loyalty_data = pd.read_csv(loyalty_data_path, encoding='latin1')
gps_data = pd.read_csv(gps_data_path, encoding='latin1')
car_assignments = pd.read_csv(car_assignments_path, encoding='latin1')

In [33]:
# Fix column names (normalize all lowercase for consistency)
cc_data.columns = cc_data.columns.str.lower()
loyalty_data.columns = loyalty_data.columns.str.lower()
gps_data.columns = gps_data.columns.str.lower()
car_assignments.columns = car_assignments.columns.str.lower()

# Convert timestamp columns to datetime
cc_data['timestamp'] = pd.to_datetime(cc_data['timestamp'], errors='coerce')
loyalty_data['timestamp'] = pd.to_datetime(loyalty_data['timestamp'], errors='coerce')
gps_data['timestamp'] = pd.to_datetime(gps_data['timestamp'], errors='coerce')

In [34]:
# Add hour, minute, second
# Extract date and hour for easier grouping
cc_data['date'] = cc_data['timestamp'].dt.date
cc_data['hour'] = cc_data['timestamp'].dt.hour
cc_data['minute'] = cc_data['timestamp'].dt.minute

loyalty_data['date'] = loyalty_data['timestamp'].dt.date

gps_data['date'] = gps_data['timestamp'].dt.date
gps_data['hour'] = gps_data['timestamp'].dt.hour
gps_data['minute'] = gps_data['timestamp'].dt.minute
gps_data['second'] = gps_data['timestamp'].dt.second



# Add dayofweek and is_weekend to cc_data, loyalty_data, and gps_data
# Add to cc_data
cc_data['dayofweek'] = cc_data['timestamp'].dt.day_name()        # e.g., 'Monday'
cc_data['is_weekend'] = cc_data['dayofweek'].isin(['Saturday', 'Sunday'])

# Add to loyalty_data (uses only date, so cast as datetime if needed)
loyalty_data['dayofweek'] = pd.to_datetime(loyalty_data['date']).dt.day_name()
loyalty_data['is_weekend'] = loyalty_data['dayofweek'].isin(['Saturday', 'Sunday'])

# Add to gps_data
gps_data['dayofweek'] = gps_data['timestamp'].dt.day_name()
gps_data['is_weekend'] = gps_data['dayofweek'].isin(['Saturday', 'Sunday'])

In [35]:
# Check that everything looks right
print("Credit Card Data:")
display(cc_data.head())

print("Loyalty Card Data:")
display(loyalty_data.head())

print("GPS Data:")
display(gps_data.head())

print("Car Assignments:")
display(car_assignments.head())

Credit Card Data:


,timestamp,location,price,last4ccnum,date,hour,minute,dayofweek,is_weekend
0,2014-01-06 07:28:00,Brew've Been Served,11.34,4795,2014-01-06,7,28,Monday,False
1,2014-01-06 07:34:00,Hallowed Grounds,52.22,7108,2014-01-06,7,34,Monday,False
2,2014-01-06 07:35:00,Brew've Been Served,8.33,6816,2014-01-06,7,35,Monday,False
3,2014-01-06 07:36:00,Hallowed Grounds,16.72,9617,2014-01-06,7,36,Monday,False
4,2014-01-06 07:37:00,Brew've Been Served,4.24,7384,2014-01-06,7,37,Monday,False


Loyalty Card Data:


,timestamp,location,price,loyaltynum,date,dayofweek,is_weekend
0,2014-01-06,Brew've Been Served,4.17,L2247,2014-01-06,Monday,False
1,2014-01-06,Brew've Been Served,9.60,L9406,2014-01-06,Monday,False
2,2014-01-06,Hallowed Grounds,16.53,L8328,2014-01-06,Monday,False
3,2014-01-06,Coffee Shack,11.51,L6417,2014-01-06,Monday,False
4,2014-01-06,Hallowed Grounds,12.93,L1107,2014-01-06,Monday,False


GPS Data:


,timestamp,id,lat,long,date,hour,minute,second,dayofweek,is_weekend
0,2014-01-06 06:28:01,35,36.076225,24.874689,2014-01-06,6,28,1,Monday,False
1,2014-01-06 06:28:01,35,36.076220,24.874596,2014-01-06,6,28,1,Monday,False
2,2014-01-06 06:28:03,35,36.076211,24.874443,2014-01-06,6,28,3,Monday,False
3,2014-01-06 06:28:05,35,36.076217,24.874253,2014-01-06,6,28,5,Monday,False
4,2014-01-06 06:28:06,35,36.076214,24.874167,2014-01-06,6,28,6,Monday,False


Car Assignments:


,lastname,firstname,carid,currentemploymenttype,currentemploymenttitle
0,Calixto,Nils,1.0,Information Technology,IT Helpdesk
1,Azada,Lars,2.0,Engineering,Engineer
2,Balas,Felix,3.0,Engineering,Engineer
3,Barranco,Ingrid,4.0,Executive,SVP/CFO
4,Baza,Isak,5.0,Information Technology,IT Technician


### Data Cleaning Summary: Type Standardization

To ensure our datasets are clean, consistent, and compatible with Tableau and our Python-based analysis, we performed the following type standardizations across all files:

---

#### **1. Credit Card Data (`cc_data.csv`)**

- **`timestamp` → `datetime64`**  
  Parsed to ensure proper time-based filtering and grouping in Tableau.

- **`location` → `string`**  
  Prevents Tableau from misinterpreting business names that contain numbers.

- **`price` → `float`**  
  Ensures numeric accuracy and allows for aggregations like average spend.

- **`last4ccnum` → `string`**  
  Treated as text to preserve leading zeros and avoid numeric formatting (e.g., 0001 → 1).

- **`hour` → `Int64`**  
  Allows for nulls while maintaining integer-based hour groupings.

- **`date` → `datetime64`**  
  Ensures compatibility with Tableau’s date functions.

---

#### **2. Loyalty Card Data (`loyalty_data.csv`)**

- **Similar changes as `cc_data`**, with `loyaltynum` also cast to `string` to preserve format and support joins.

---

#### **3. GPS Vehicle Data (`gps.csv`)**

- **`timestamp` → `datetime64`**  
  For time-aligned matching with transactions.

- **`id` → `string`**  
  Treated as an identifier rather than a numeric value.

- **`lat` / `long` → `float64`**  
  Needed for map plotting and distance calculations.

- **`hour` → `Int64`**  
  Keeps hourly analysis consistent with other datasets.

- **`date` → `datetime64`**  
  For time-based joins and filtering.

---

#### **4. Car Assignment Data (`car-assignments.csv`)**

- **`firstname`, `lastname`, `currentemploymenttype`, `currentemploymenttitle` → `string`**  
  Cleaned for easier matching and labeling in visualizations.

- **`carid` → `string`**  
  Cast as text to avoid issues with formatting or merging with GPS data where it might be string-based.

---

### Why This Matters

Standardizing data types is critical to:
- Prevent Tableau from auto-detecting incorrect field types (e.g., IDs as numbers).
- Ensure joins and merges behave predictably across datasets.
- Enable consistent use of filters, groupings, and calculations.
- Avoid silent errors when processing datetime or mixed-type fields in Python.

In [36]:
# --------------------------
# Type Cleaning: cc_data
# --------------------------
cc_data['timestamp'] = pd.to_datetime(cc_data['timestamp'], errors='coerce')
cc_data['location'] = cc_data['location'].astype('string')
cc_data['price'] = pd.to_numeric(cc_data['price'], errors='coerce')
cc_data['last4ccnum'] = cc_data['last4ccnum'].astype('string') # Keep as string to avoid 0000 → 0
cc_data['dayofweek'] = cc_data['dayofweek'].astype('string')  
cc_data['hour'] = cc_data['hour'].astype('Int64')  # nullable int
cc_data['minute'] = cc_data['minute'].astype('Int64')  # nullable int
cc_data['date'] = pd.to_datetime(cc_data['date'], errors='coerce')

# -------------------------------
# Type Cleaning: loyalty_data
# -------------------------------
loyalty_data['timestamp'] = pd.to_datetime(loyalty_data['timestamp'], errors='coerce')
loyalty_data['location'] = loyalty_data['location'].astype('string')
loyalty_data['price'] = pd.to_numeric(loyalty_data['price'], errors='coerce')
loyalty_data['loyaltynum'] = loyalty_data['loyaltynum'].astype('string')
loyalty_data['dayofweek'] = loyalty_data['dayofweek'].astype('string')
loyalty_data['date'] = pd.to_datetime(loyalty_data['date'], errors='coerce')

# --------------------------
# Type Cleaning: gps_data
# --------------------------
gps_data['timestamp'] = pd.to_datetime(gps_data['timestamp'], errors='coerce')
gps_data['id'] = gps_data['id'].astype('string')
gps_data['lat'] = pd.to_numeric(gps_data['lat'], errors='coerce')
gps_data['long'] = pd.to_numeric(gps_data['long'], errors='coerce')
gps_data['hour'] = gps_data['hour'].astype('Int64')
gps_data['minute'] = gps_data['minute'].astype('Int64')
gps_data['second'] = gps_data['second'].astype('Int64')
gps_data['date'] = pd.to_datetime(gps_data['date'], errors='coerce')
gps_data['dayofweek'] = gps_data['dayofweek'].astype('string')

# -------------------------------
# Type Cleaning: car_assignments
# -------------------------------
car_assignments['firstname'] = car_assignments['firstname'].astype('string')
car_assignments['lastname'] = car_assignments['lastname'].astype('string')
car_assignments['carid'] = car_assignments['carid'].astype('string')
car_assignments['currentemploymenttype'] = car_assignments['currentemploymenttype'].astype('string')
car_assignments['currentemploymenttitle'] = car_assignments['currentemploymenttitle'].astype('string')

### Column Linkages Across Datasets


This summary outlines how columns from different datasets relate to one another and what role each plays in our analysis and visualizations. Now that all object fields have been converted to explicit `string` types and all `date` fields are properly cast as `datetime64`, we can ensure consistent merging, filtering, and grouping across all tools — especially Tableau.

---

####   `timestamp` (Primary Temporal Key)
- **Present In**: `cc_data`, `loyalty_data`, `gps_data`
- **Type**: `datetime64[ns]`
- **Purpose**: Core time field to align activities (transactions, movement) across datasets.
- **Use Cases**:
  - Match vehicle presence to transaction time (Questions 2, 3, 5)
  - Group events by time of day, weekday, or window (heatmaps, time-aligned maps)

---

####   `location`
- **Present In**: `cc_data`, `loyalty_data`
- **Type**: `string`
- **Purpose**: Identifies the business visited.
- **Use Cases**:
  - Determine popular locations (Question 1)
  - Detect anomalies and suspicious stops (Questions 1, 2, 5)
  - Join GPS coordinates or map metadata manually if geocoded

---

####   `price`
- **Present In**: `cc_data`, `loyalty_data`
- **Type**: `float64`
- **Purpose**: Tracks amount spent at each location.
- **Use Cases**:
  - Detect outlier purchases (unusual transactions)
  - Validate whether a stop was legitimate (vs. idle with no purchase)

---

####   `last4ccnum` / `loyaltynum`
- **Present In**: `cc_data` (`last4ccnum`), `loyalty_data` (`loyaltynum`)
- **Type**: `string`
- **Purpose**: Anonymized card IDs.
- **Use Cases**:
  - Infer card ownership (Question 3)
  - Group repeat patterns by card activity
  - Link back to GPS patterns and employee IDs via timestamp correlation

---

####   `date`, `hour`, `minute`, `second`, `dayofweek`, `is_weekend`
- **Present In**: All time-based datasets
- **Type**: `datetime64`, `int32`, `string`, `bool`
- **Purpose**: Breaks timestamp into analyzable chunks.
- **Use Cases**:
  - Group by time of day or weekday (Questions 1, 5)
  - Visualize usage heatmaps
  - Flag anomalies during weekends or off-hours

---

####   `id` (GPS vehicle ID) ↔ `carid` (Car Assignments)
- **Present In**: `gps_data` (`id`), `car_assignments` (`carid`)
- **Type**: `string`
- **Purpose**: Core link between vehicle GPS logs and employee information.
- **Use Cases**:
  - Attribute movement to specific employees (Questions 2, 3, 5)
  - Detect behavior patterns by role or department (Question 4)

---

####   `firstname`, `lastname`, `currentemploymenttype`, `currentemploymenttitle`
- **Present In**: `car_assignments`
- **Type**: `string`
- **Purpose**: Describes who is assigned to a car and their job role.
- **Use Cases**:
  - Add human-readable labels for filtering and dashboards
  - Detect relationship clusters (Question 4)
  - Enhance final report readability

---

###   Summary Table by Question

| Question | Columns Used | Description |
|----------|--------------|-------------|
| **Q1** – Most popular locations | `timestamp`, `location`, `price`, `hour`, `dayofweek` | Identify top locations and peak hours |
| **Q2** – Add vehicle data | `timestamp`, `id`, `location`, `lat`, `long` | Match purchases to vehicle stops |
| **Q3** – Card ownership | `last4ccnum`, `loyaltynum`, `id`, `carid`, `timestamp`, `location` | Infer which cards belong to which people |
| **Q4** – Informal relationships | `timestamp`, `id`, `carid`, `firstname`, `location`, `lat`, `long` | Identify co-locations and social patterns |
| **Q5** – Suspicious activity | `timestamp`, `location`, `lat`, `long`, `price`, `id`, `carid` | Detect unusual patterns, idle times, and unauthorized visits |



### Output Cleaned Data in "/cleaned_data"

In [37]:
for name, df in {
    "CC Data": cc_data,
    "Loyalty Data": loyalty_data,
    "GPS Data": gps_data,
    "Car Assignments": car_assignments
}.items():
    print(f"\n{name} Columns and Types:\n")
    print(df.dtypes)


CC Data Columns and Types:

timestamp     datetime64[ns]
location      string[python]
price                float64
last4ccnum    string[python]
date          datetime64[ns]
hour                   Int64
minute                 Int64
dayofweek     string[python]
is_weekend              bool
dtype: object

Loyalty Data Columns and Types:

timestamp     datetime64[ns]
location      string[python]
price                float64
loyaltynum    string[python]
date          datetime64[ns]
dayofweek     string[python]
is_weekend              bool
dtype: object

GPS Data Columns and Types:

timestamp     datetime64[ns]
id            string[python]
lat                  float64
long                 float64
date          datetime64[ns]
hour                   Int64
minute                 Int64
second                 Int64
dayofweek     string[python]
is_weekend              bool
dtype: object

Car Assignments Columns and Types:

lastname                  string[python]
firstname                 string[

In [38]:
# Step 1: Define output directory name
output_dir = "cleaned_data"

# Step 2: Create the directory if it doesn't already exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")
else:
    print(f"Directory already exists: {output_dir}")

# Step 3: Save each cleaned DataFrame as a CSV
cc_data.to_csv(f"{output_dir}/cc_data_cleaned.csv", index=False)
loyalty_data.to_csv(f"{output_dir}/loyalty_data_cleaned.csv", index=False)
gps_data.to_csv(f"{output_dir}/gps_data_cleaned.csv", index=False)
car_assignments.to_csv(f"{output_dir}/car_assignments_cleaned.csv", index=False)

print("Cleaned files saved successfully!")

Directory already exists: cleaned_data
Cleaned files saved successfully!
